# 01b — Hazard Exposure from COGs on Azure (prototype, no Earth Engine compute)

Same partials as notebook 01 (`s01`) and `s01c`, computed outside Earth Engine: the WorldPop R2025A
population and the GCHD hazard rasters are read as **Cloud-Optimized GeoTIFFs (COGs)** from the unit's
Azure storage and reduced per admin-2 unit with **exactextract** (open source). Runs the same on a laptop
or in Databricks; the code is `pipeline/steps/s01b.py` and `pipeline/cog_zonal.py`.

## How it works, per country

1. **Boundaries** — the country's admin-2 units from `global_boundary/admin2_pop_geom`, cached once as
   GeoParquet in `CACHE_DIR/boundaries/`. Earth Engine is only called when that file is missing.
2. **Population** — the six COGs (`T, T_F, T_M, T_U18, F_U18, M_U18`) under
   `public/worldpop_2025/R2025A/2025/100m/`, either downloaded once (`download`) or read in place (`remote`).
3. **Hazards** — every COG under `public/gchd_cog/` is read for the country's extent only, and warped
   (nearest neighbour) onto the population grid, as Earth Engine does when it reduces at the population scale.
4. **Zonal statistics** — exactextract weights each pixel by the share of it inside the unit: six sums of
   population × threshold mask (`exp_*`), and mean, min, max, median and std of the raw hazard.
5. **Output** — the partial CSVs, identical in layout to `s01`, so notebooks 02–05 run unchanged.

## COGs in two lines

A COG is a GeoTIFF whose tiles (512 × 512 here) and index are laid out so a reader fetches only the tiles
it needs with HTTP range requests. Reading one country out of a global hazard raster therefore moves
megabytes, not the whole file. Overviews exist for maps; statistics always read full resolution.

## Where to run it

| | Local laptop | Databricks |
|---|---|---|
| `POPULATION` | `"download"` (whole-file download is faster than many range reads over a slow link) | `"remote"` (next to the storage, reads are fast and free of egress) |
| `WORKERS` | 1–2 (CPU and link bound) | one per core; scale the cluster for more countries |
| `STORAGE_ROOT` | `notebooks/output` | a Unity Catalog volume, e.g. `/Volumes/<catalog>/<schema>/<volume>/gchd` |
| Boundaries | fetched from Earth Engine on first use | upload `CACHE_DIR/boundaries/*.parquet` once, or set a service-account key |

## Hazards covered

Only hazards whose COG is the same raster as the GEE asset (`cog_zonal.COG_LAYERS`, 14 codes).
Heat, SPI/SPEI, drought ASI and river flood COGs are different products and are left to `s01`/`s01c`.
Tropical storm and malaria Pv statistics differ from GEE in the benchmark and are under review.

Special population rasters (Alaska, Åland, Kosovo) are not handled yet.

In [ ]:
# ============================================================
# Configuration
# ============================================================
COUNTRIES = ["MWI", "BDI"]      # ISO3 or ucodes; required (prototype)
HAZARDS = None                  # None -> every hazard in cog_zonal.COG_LAYERS
POPULATION = None               # None -> "remote" on Databricks, "download" elsewhere
WORKERS = None                  # None -> one per core on Databricks, 1 elsewhere
STORAGE_ROOT = None             # None -> notebooks/output (or GCHD_STORAGE_ROOT)
CACHE_DIR = None                # None -> <storage_root>/cache (boundaries + downloaded population)
MAX_CACHE_GB = 20               # disk the downloaded population may hold at once
KEEP_CACHE = False              # keep each country's population files after use

## Databricks only — install the packages

The cluster needs `exactextract`, `rasterio`, `geopandas` and `pyarrow`, plus this repository
(a Git folder in the workspace). Run the next cell once per cluster session; it is skipped elsewhere.

In [ ]:
import os
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
if ON_DATABRICKS:
    get_ipython().run_line_magic("pip", "install -q exactextract==0.3.0 rasterio geopandas pyarrow earthengine-api pydantic-settings")

In [ ]:
# ============================================================
# Imports and settings
# ============================================================
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline" / "settings.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from pipeline.settings import Settings
from pipeline.steps import s01b
from pipeline import io_paths

POPULATION = POPULATION or ("remote" if ON_DATABRICKS else "download")
WORKERS = WORKERS or ((os.cpu_count() or 2) if ON_DATABRICKS else 1)
overrides = {"process_countries": COUNTRIES}
if STORAGE_ROOT:
    overrides["storage_root"] = STORAGE_ROOT
if HAZARDS:
    overrides["process_hazards"] = HAZARDS
settings = Settings.from_env(**overrides)
print(f"storage_root={settings.storage_root}  population={POPULATION}  workers={WORKERS}")

## Optional — look inside one COG

`gdalinfo`-style view of a population COG read over HTTP: block size, overviews and compression.
Only the header is fetched.

In [ ]:
import rasterio
from pipeline.cog_zonal import pop_url
with rasterio.open(f"/vsicurl/{pop_url(COUNTRIES[0][:3], 'total', 2025)}") as src:
    print(src.profile)
    print("blocks:", src.block_shapes, " overviews:", src.overviews(1))

In [ ]:
# ============================================================
# Run
# ============================================================
result = s01b.run(settings, cache_dir=CACHE_DIR, population=POPULATION, workers=WORKERS,
                  max_cache_gb=MAX_CACHE_GB, keep_cache=KEEP_CACHE)
result

In [ ]:
# ============================================================
# Timing per country (compute_s is the zonal statistics; population_s the download)
# ============================================================
pd.read_csv(Path(io_paths.manifest_root(settings)) / "s01b_timing.csv")

## Next

The partials are in `{storage_root}/base/hazards/{code}/partials/`. Continue with notebook 02
(indices and NA rules) on the same `storage_root`.